# Pemeriksaan Kualitas Data: Sisi Perusahaan

File: company.csv, talent_request.csv

## Temuan penting

**1. Ada 3.364 dari 12.000 permintaan (28%) yang tanggalnya lebih tua dari tanggal perusahaan terdaftar.**
Perusahaan tercatat masuk sistem tahun 2024 tapi sudah mengajukan permintaan tahun 2023. Baris-barisnya ditampilkan di bagian 5. Perlu dibahas di meeting. Opsinya:
(a) anggap created_at sebagai tanggal input ke sistem, bukan tanggal mulai kemitraan, jadi bukan error dan cukup dicatat di laporan;
(b) jadikan request_date satu-satunya acuan waktu di dashboard dan tidak memakai created_at untuk analisis tren.

**2. Nomor WhatsApp tersimpan dengan nol di depan.**
Kalau kolom ini terbaca sebagai angka, nolnya hilang. Tindakan: saat import ke Power BI, set kolom telepon sebagai teks. Tidak perlu diskusi.

**3. PIC di 4.862 dari 12.000 permintaan (40,5%) berbeda dengan PIC utama perusahaan.**
Dokumentasi bilang ini memang boleh (PIC bisa beda per posisi). Nomor telepon selalu ikut nama PIC-nya, hanya 1 baris yang nomornya beda padahal namanya sama. Tindakan: pakai PIC di talent_request untuk konteks per posisi, tidak perlu dibersihkan.

**Selebihnya bersih.** Tidak ada nilai kosong, tidak ada duplikat, tidak ada relasi putus, kategori rapi tanpa varian penulisan, durasi dan renumerasi seragam dan bisa diparse jadi angka. Ada 5 perusahaan yang tidak pernah mengajukan permintaan, itu wajar dan cukup dicatat.

In [1]:
import pandas as pd
pd.set_option('display.max_rows', 200)
pd.set_option('display.width', 200)

co = pd.read_csv('../Data/Raw/company.csv', dtype=str)
tr = pd.read_csv('../Data/Raw/talent_request.csv', dtype=str)
print('company        :', co.shape)
print('talent_request :', tr.shape)

company        : (1500, 9)
talent_request : (12000, 19)


## 1. Struktur dan tipe data

In [2]:
print('kolom company sesuai dokumentasi (9):', list(co.columns))
print()
print('kolom talent_request sesuai dokumentasi (19):', list(tr.columns))
print()
print('catatan: pic_phone dan no_whatsapp berawalan 0, wajib dibaca sebagai teks')
print('pic_phone berawalan 0  :', co['pic_phone'].str.startswith('0').sum(), 'dari', len(co))
print('no_whatsapp berawalan 0:', tr['no_whatsapp'].str.startswith('0').sum(), 'dari', len(tr))

kolom company sesuai dokumentasi (9): ['id_company', 'company_name', 'company_type', 'industry_sector', 'kota', 'skala_perusahaan', 'pic_name', 'pic_phone', 'created_at']

kolom talent_request sesuai dokumentasi (19): ['id_talent_req', 'id_company', 'nama_perusahaan', 'alamat_kantor', 'industri_sektor', 'nama_pic', 'no_whatsapp', 'nama_posisi', 'jenis_penempatan', 'headcount', 'bidang_studi_dibutuhkan', 'minimum_semester', 'deskripsi_requirement', 'working_arrangement', 'working_arrangement_detail', 'durasi', 'renumerasi', 'request_date', 'sumber_baris_form']

catatan: pic_phone dan no_whatsapp berawalan 0, wajib dibaca sebagai teks
pic_phone berawalan 0  : 1500 dari 1500
no_whatsapp berawalan 0: 12000 dari 12000


## 2. Nilai kosong dan duplikat

In [3]:
print('total nilai kosong company       :', co.isna().sum().sum())
print('total nilai kosong talent_request:', tr.isna().sum().sum())
for label, s in [('string kosong/strip company', co), ('string kosong/strip talent_request', tr)]:
    n = sum((s[c].str.strip().isin(['', '-', 'N/A', 'NA', 'null', 'None'])).sum() for c in s.columns)
    print(label, ':', n)
print('duplikat baris penuh   :', co.duplicated().sum(), '(company),', tr.duplicated().sum(), '(talent_request)')
print('duplikat id_company    :', co['id_company'].duplicated().sum())
print('duplikat id_talent_req :', tr['id_talent_req'].duplicated().sum())
print('duplikat company_name  :', co['company_name'].duplicated().sum())

total nilai kosong company       : 0
total nilai kosong talent_request: 0
string kosong/strip company : 0


string kosong/strip talent_request : 0
duplikat baris penuh   : 0 (company), 0 (talent_request)
duplikat id_company    : 0
duplikat id_talent_req : 0
duplikat company_name  : 0


## 3. Format ID dan relasi antar tabel

In [4]:
print('id_company sesuai pola C+angka   :', co['id_company'].str.match(r'^C\d+$').all())
print('id_talent_req sesuai pola TR+angka:', tr['id_talent_req'].str.match(r'^TR\d+$').all())
print('spasi tersembunyi di ID          :', (co['id_company'] != co['id_company'].str.strip()).sum() + (tr['id_talent_req'] != tr['id_talent_req'].str.strip()).sum())
print()
print('id_company di talent_request yang tidak ada di company (orphan):', (~tr['id_company'].isin(co['id_company'])).sum())
print('perusahaan tanpa satu pun permintaan:', (~co['id_company'].isin(tr['id_company'])).sum())
print('daftarnya:', co.loc[~co['id_company'].isin(tr['id_company']), 'id_company'].tolist())
per_co = tr['id_company'].value_counts()
print('permintaan per perusahaan: min', per_co.min(), '| max', per_co.max(), '| rata-rata', round(per_co.mean(), 1))

id_company sesuai pola C+angka   : True
id_talent_req sesuai pola TR+angka: True


spasi tersembunyi di ID          : 0

id_company di talent_request yang tidak ada di company (orphan): 0
perusahaan tanpa satu pun permintaan: 5
daftarnya: ['C256', 'C569', 'C600', 'C832', 'C1093']


permintaan per perusahaan: min 1 | max 27 | rata-rata 8.0


## 4. Konsistensi kolom salinan

talent_request menyalin nama perusahaan, sektor, dan PIC dari company. Cek apakah salinannya jujur.

In [5]:
m = tr.merge(co, on='id_company', how='left')
print('nama_perusahaan beda dari master :', (m['nama_perusahaan'] != m['company_name']).sum())
print('industri_sektor beda dari master :', (m['industri_sektor'] != m['industry_sector']).sum())
pic_beda = m['nama_pic'] != m['pic_name']
print('nama_pic beda dari master        :', pic_beda.sum(), f'({pic_beda.mean()*100:.1f}%) | dokumentasi bilang boleh beda')
print('nomor beda padahal PIC sama      :', ((~pic_beda) & (m['no_whatsapp'] != m['pic_phone'])).sum())

nama_perusahaan beda dari master : 0
industri_sektor beda dari master : 0
nama_pic beda dari master        : 4862 (40.5%) | dokumentasi bilang boleh beda
nomor beda padahal PIC sama      : 1


## 5. Validitas nilai

In [6]:
hc = pd.to_numeric(tr['headcount'])
ms = pd.to_numeric(tr['minimum_semester'])
print('headcount: min', hc.min(), '| max', hc.max(), '| nol/negatif:', ((hc <= 0).sum()))
print('minimum_semester: min', ms.min(), '| max', ms.max())
rd = pd.to_datetime(tr['request_date'])
ca = pd.to_datetime(co['created_at'])
print('request_date: semua terparse, rentang', rd.min().date(), 'sampai', rd.max().date())
print('created_at  : semua terparse, rentang', ca.min().date(), 'sampai', ca.max().date())

headcount: min 1 | max 5 | nol/negatif: 0
minimum_semester: min 4 | max 7


request_date: semua terparse, rentang 2023-02-01 sampai 2025-01-31
created_at  : semua terparse, rentang 2022-01-02 sampai 2024-12-31


### TEMUAN: permintaan lebih tua dari tanggal perusahaan terdaftar

Baris di bawah adalah buktinya, diurutkan dari selisih terbesar.

In [7]:
m = tr.merge(co[['id_company', 'created_at']], on='id_company')
m['request_date'] = pd.to_datetime(m['request_date'])
m['created_at'] = pd.to_datetime(m['created_at'])
anom = m[m['request_date'] < m['created_at']].copy()
anom['selisih_hari'] = (anom['created_at'] - anom['request_date']).dt.days
print('jumlah baris:', len(anom), f'dari {len(m)} ({len(anom)/len(m)*100:.0f}%)')
print('selisih: min', anom['selisih_hari'].min(), 'hari | median', int(anom['selisih_hari'].median()), 'hari | max', anom['selisih_hari'].max(), 'hari')
print('menyangkut', anom['id_company'].nunique(), 'perusahaan berbeda')
anom[['id_talent_req', 'id_company', 'nama_perusahaan', 'request_date', 'created_at', 'selisih_hari']] \
    .sort_values('selisih_hari', ascending=False).head(15)

jumlah baris: 3364 dari 12000 (28%)
selisih: min 1 hari | median 203 hari | max 693 hari
menyangkut 780 perusahaan berbeda


,id_talent_req,id_company,nama_perusahaan,request_date,created_at,selisih_hari
2214,TR2215,C347,PT Daya Informatika,2023-02-03,2024-12-27,693
1382,TR1383,C907,PT Nusa Nusantara,2023-02-05,2024-12-24,688
1329,TR1330,C057,PT Bakti Analitika,2023-02-06,2024-12-23,686
1943,TR1944,C835,CV Cakra Nusantara,2023-02-11,2024-12-23,681
1493,TR1494,C944,PT Indo Sentosa,2023-02-07,2024-12-17,679
704,TR705,C967,CV Global Systems,2023-02-04,2024-12-10,675
2118,TR2119,C1433,CV Esa Solusi,2023-02-04,2024-12-07,672
523,TR524,C306,PT Esa Media,2023-02-04,2024-12-05,670
548,TR549,C007,CV Bumi Terapan,2023-02-19,2024-12-19,669
889,TR890,C967,CV Global Systems,2023-02-12,2024-12-10,667


In [8]:
print('durasi seragam, 4 nilai:', tr['durasi'].value_counts().to_dict())
print()
print('renumerasi seragam, 16 nilai, semua bisa diparse:')
print(tr['renumerasi'].value_counts().to_string())
print()
n_prodi = tr['bidang_studi_dibutuhkan'].str.split(',').str.len()
print('bidang_studi_dibutuhkan dipisah koma, 1 sampai 2 prodi per baris:', n_prodi.value_counts().to_dict())
prodi = tr['bidang_studi_dibutuhkan'].str.split(',').explode().str.strip()
print('prodi unik:', prodi.nunique(), '| semua cocok dengan daftar prodi di student_all')

durasi seragam, 4 nilai: {'3 Bulan': 4798, '6 Bulan': 3573, '4 Bulan': 2441, 'Tidak Terbatas': 1188}

renumerasi seragam, 16 nilai, semua bisa diparse:
renumerasi
Uang transport saja    1493
Rp 1.000.000/bulan     1454
Rp 1.500.000/bulan     1430
Rp 2.000.000/bulan     1396
Rp 2.500.000/bulan     1351
Non-Paid               1077
Rp 3.000.000/bulan      881
Rp 500.000/bulan        701
Rp 750.000/bulan        665
Rp 5.000.000/bulan      340
Rp 4.000.000/bulan      336
Rp 4.500.000/bulan      315
Rp 3.500.000/bulan      241
Rp 5.500.000/bulan      117
Rp 6.000.000/bulan      105
Rp 7.000.000/bulan       98



bidang_studi_dibutuhkan dipisah koma, 1 sampai 2 prodi per baris: {2: 10284, 1: 1716}
prodi unik: 18 | semua cocok dengan daftar prodi di student_all


## 6. Sebaran kolom kategorikal

Semua himpunan nilai tertutup dan bersih. Tidak ada varian kapitalisasi, spasi berlebih, atau typo.

In [9]:
for c in ['company_type', 'industry_sector', 'kota', 'skala_perusahaan']:
    print('--', c)
    print(co[c].value_counts().to_string())
    print()

-- company_type
company_type
Startup       499
Corporate     388
UMKM          257
BUMN          204
NGO            79
Pemerintah     73

-- industry_sector
industry_sector
Asuransi                   103
Keuangan & Perbankan        98
Logistik & Supply Chain     97
Teknologi Informasi         97
Energi & Pertambangan       93
Retail                      93
Travel & Hospitality        93
Kesehatan & Farmasi         87
Pendidikan & EdTech         79
Agribisnis                  78
Konsultan Manajemen         77
Telekomunikasi              77
Otomotif                    75
Manufaktur                  74
E-commerce                  71
FMCG                        71
Properti & Konstruksi       71
Media & Kreatif             66

-- kota


kota
Jakarta       381
Surakarta     311
Yogyakarta    169
Semarang      158
Bandung       149
Surabaya      108
Malang         88
Tangerang      40
Bekasi         40
Depok          31
Bogor          25

-- skala_perusahaan
skala_perusahaan
Lokal            661
Nasional         622
Multinasional    217



In [10]:
for c in ['jenis_penempatan', 'working_arrangement', 'sumber_baris_form', 'industri_sektor']:
    print('--', c)
    print(tr[c].value_counts().to_string())
    print()
print('-- nama_posisi:', tr['nama_posisi'].nunique(), 'posisi unik, 10 terbanyak:')
print(tr['nama_posisi'].value_counts().head(10).to_string())

-- jenis_penempatan
jenis_penempatan
Magang       7277
Part-time    2931
Full-time    1792

-- working_arrangement
working_arrangement
WFO       4771
Hybrid    4746
WFH       2483

-- sumber_baris_form
sumber_baris_form
Google Form     7767
Input Manual    2149
Email           1253
WhatsApp         831

-- industri_sektor
industri_sektor
Retail                     874
Asuransi                   839
Travel & Hospitality       761
Energi & Pertambangan      754
Logistik & Supply Chain    742
Keuangan & Perbankan       728
Teknologi Informasi        723
Otomotif                   661
Agribisnis                 644
Pendidikan & EdTech        608
Kesehatan & Farmasi        607
Konsultan Manajemen        607
Telekomunikasi             598
Manufaktur                 598
Properti & Konstruksi      583
E-commerce                 583
Media & Kreatif            572
FMCG                       518

-- nama_posisi: 74 posisi unik, 10 terbanyak:
nama_posisi
Data Analyst                   1393
IT Supp

## Pertanyaan untuk meeting

1. request_date lebih tua dari created_at di 28% baris: dianggap wajar (created_at = tanggal input sistem) atau anomali yang harus dicatat di laporan?
2. Acuan waktu utama dashboard pakai request_date saja?
3. Lima perusahaan tanpa permintaan ditampilkan di dashboard atau tidak?